# Phase I · Hypothesis 2: Word clusters and model behaviour

Lukas showed in hypothesis 1 how much the irony score depends on `#irony`, `#sarcasm` and `#not`. This notebook looks at the rest of the text.

**Hypothesis:** The model treats certain words or topics as an irony signal. In groups of tweets with those words, the mean irony score should then be clearly higher or lower than the share of tweets that really are ironic.

In [1]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import snapshot_download
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = "cardiffnlp/twitter-roberta-base-irony"
MODEL_REVISION = "3bf8f118bdf6b00c99658151ef10c9a0b9afd6bf"

# show the full tweet text in tables
pd.set_option("display.max_colwidth", None)

## 1. Data and scores

In [2]:
tweets = load_dataset("cardiffnlp/tweet_eval", "irony", split="test").to_pandas()
print(tweets.shape)
tweets.head()

(784, 2)


,text,label
0,@user Can U Help?||More conservatives needed on #TSU + get paid 4 posting stuff like this!||YOU $ can go to,0
1,"Just walked in to #Starbucks and asked for a ""tall blonde"" Hahahaha #irony",1
2,#NOT GONNA WIN,0
3,@user He is exactly that sort of person. Weirdo!,0
4,So much #sarcasm at work mate 10/10 #boring 100% #dead mate full on #shit absolutely #sleeping mate can't handle the #sarcasm,1


Loading and scoring are taken from Lukas' notebook, so both hypotheses use the same scores.

In [3]:
checkpoint = snapshot_download(
    MODEL_ID,
    revision=MODEL_REVISION,
    allow_patterns=["config.json", "pytorch_model.bin", "merges.txt",
                    "vocab.json", "special_tokens_map.json"],
)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
model.eval()


def normalize_tweet(text):
    tokens = []
    for token in text.split():
        if token.startswith("@") and len(token) > 1:
            token = "@user"
        elif token.startswith("http"):
            token = "http"
        tokens.append(token)
    return " ".join(tokens)


@torch.inference_mode()
def irony_scores(texts, batch_size=16):
    scores = []
    for start in range(0, len(texts), batch_size):
        batch = [normalize_tweet(text) for text in texts[start:start + batch_size]]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=512, return_tensors="pt")
        logits = model(**inputs).logits
        scores.extend(logits.softmax(dim=-1)[:, 1].tolist())
    return scores

In [4]:
tweets["score"] = irony_scores(tweets["text"].tolist())
tweets["prediction"] = (tweets["score"] >= 0.5).astype(int)

tweets.head()

,text,label,score,prediction
0,@user Can U Help?||More conservatives needed on #TSU + get paid 4 posting stuff like this!||YOU $ can go to,0,0.303482,0
1,"Just walked in to #Starbucks and asked for a ""tall blonde"" Hahahaha #irony",1,0.649670,1
2,#NOT GONNA WIN,0,0.032981,0
3,@user He is exactly that sort of person. Weirdo!,0,0.209184,0
4,So much #sarcasm at work mate 10/10 #boring 100% #dead mate full on #shit absolutely #sleeping mate can't handle the #sarcasm,1,0.074537,0


## 2. Four groups

In [5]:
ironic = tweets["label"] == 1
predicted_ironic = tweets["prediction"] == 1

tweets.loc[ironic & predicted_ironic, "group"] = "true ironic"
tweets.loc[~ironic & ~predicted_ironic, "group"] = "true non-ironic"
tweets.loc[~ironic & predicted_ironic, "group"] = "false ironic"
tweets.loc[ironic & ~predicted_ironic, "group"] = "false non-ironic"

tweets["error"] = tweets["label"] != tweets["prediction"]

print("Accuracy:", round(1 - tweets["error"].mean(), 3))
tweets["group"].value_counts()

Accuracy: 0.735


group
true non-ironic     401
true ironic         175
false non-ironic    136
false ironic         72
Name: count, dtype: int64

The groups differ a lot in size. *False ironic* has only 72 tweets, so its word counts need care. The model also misses irony (136) almost twice as often as it wrongly detects it (72).